# Name That LLM: which encoder learns the authors best?

**Goal:** fine-tune two transformer encoders as 18-way classifiers and keep the better one.
- **DeBERTa-v3-base** (Microsoft, 2021) is the classic workhorse of Kaggle text competitions.
- **ModernBERT-base** (Answer.AI, 2024) is newer, lighter on memory, and its byte-level
  tokenizer keeps every character.

Part of the learning-in-public MLOps project
[github.com/Ryqn0/name-that-llm](https://github.com/Ryqn0/name-that-llm), plan P5.

**What "fine-tuning" means here:** in our previous model (P3b), Qwen 3 stayed **frozen**,
and only a small logistic regression learned on top of its hidden states. Here **every
weight** of the encoder is updated for our task. That's more powerful, but it's also easier
to overfit: we have only 150 prompts.

**How we keep it honest:**
- Same prompt-grouped 5-fold CV as every other model, with the **seed-7** folds used for
  choices.
- A **fresh** pretrained model for each fold.
- A **fixed** number of epochs: no early stopping on the validation fold, which would leak
  it into the score.
- The bar to beat: TF-IDF `lr` on the same folds. The design's P5 criterion is *"an encoder
  model beats the baseline in CV"*.

In [ ]:
# Pinned to an exact commit, so this run is reproducible even after main moves on.
%pip install -q "name-that-llm[track] @ git+https://github.com/Ryqn0/name-that-llm@7cfe066b68ba28150bc738f3e26f229c1e56e3bf"

In [ ]:
import gc
import os
import time

import numpy as np
import pandas as pd
import torch
import transformers

from name_that_llm.config import EncoderConfig, load_config
from name_that_llm.data import load_train
from name_that_llm.encoder import load_encoder, run_cv_encoder, train_encoder
from name_that_llm.train import run_cv

print("torch       ", torch.__version__, "| CUDA", torch.version.cuda)
print("transformers", transformers.__version__)
print(
    "GPUs        ",
    torch.cuda.device_count(),
    [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
)
# One GPU is enough for this notebook. With two, training two folds in parallel would halve
# the time: a later idea.
assert torch.cuda.is_available(), "this notebook needs a GPU (accelerator: T4)"

# Tracking is optional: without the Kaggle Secret, everything still runs.
try:
    from kaggle_secrets import UserSecretsClient

    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    TRACK = True
except Exception as exc:
    TRACK = False
    print(f"W&B tracking is OFF ({type(exc).__name__}). Results are still computed.")

train = load_train()
SEED = 7  # the folds for CHOICES; fresh seeds (42, 43, 44) report the final numbers later
CONFIGS = {
    name: load_config(name, model_cls=EncoderConfig)
    for name in ("deberta_v3_base", "modernbert_base")
}
for cfg in CONFIGS.values():
    print(cfg)

## 1. Fail fast: minute 1, not hour 1

Before any long training, each model does **one** training step on its worst case: a full
batch of the longest training passage, at `max_length`. If the GPU memory overflows, or
the float16 loss isn't finite, we find out now. We also look at how each tokenizer cuts a
sentence with typical LLM punctuation: an em dash and curly quotes are style clues here,
so a tokenizer that keeps them is a plus.

In [ ]:
longest = max(train["text"], key=len)
SAMPLE = "It’s not just style — it’s “signature” punctuation."
for name, cfg in CONFIGS.items():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    model, tokenizer = load_encoder(cfg.base_model)
    n_params = sum(p.numel() for p in model.parameters())
    batch = [longest] * cfg.batch_size
    labels = np.array([train["model"].iloc[0]] * cfg.batch_size)
    loss = train_encoder(model, tokenizer, batch, labels, cfg.model_copy(update={"epochs": 1}))
    peak = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0
    print(
        f"{name}: {n_params / 1e6:.0f}M parameters | one step at batch {cfg.batch_size} x "
        f"{cfg.max_length} tokens: loss {loss[0]:.3f}, peak GPU memory {peak:.1f} GB"
    )
    print("   tokens:", tokenizer.tokenize(SAMPLE))
    del model
    gc.collect()
    torch.cuda.empty_cache()

## 2. The bar to beat: TF-IDF `lr` on the same seed-7 folds

In [ ]:
lr_cv = run_cv(load_config("lr").model_copy(update={"seed": SEED}), train)
print(f"TF-IDF lr on the seed-7 folds: {lr_cv.score:.4f}")

## 3. Fine-tune both encoders (5 folds each)

Each line below is one fold: a fresh pretrained model, fine-tuned on 4/5 of the prompts,
then scored on the remaining fifth. The epoch losses show the training at work. A healthy
run starts near ln(18) = 2.89 (an 18-way guess) and falls.

In [ ]:
results, rows = {}, []
for name, cfg in CONFIGS.items():
    start = time.time()
    result = run_cv_encoder(cfg.model_copy(update={"seed": SEED}), train)
    minutes = (time.time() - start) / 60
    results[name] = result
    np.save(f"/kaggle/working/oof-{name}.npy", result.oof.astype(np.float32))
    rows.append(
        {
            "config": name,
            "cv": result.score,
            "minutes": minutes,
            "fold_scores": " ".join(f"{s:.3f}" for s in result.fold_scores),
        }
    )
    print(f"==> {name}: CV {result.score:.4f} in {minutes:.0f} min", flush=True)

np.save("/kaggle/working/train_ids.npy", train["id"].to_numpy())
table = pd.DataFrame(rows).set_index("config")
table.loc["lr (TF-IDF)"] = {"cv": lr_cv.score, "minutes": np.nan, "fold_scores": ""}
# print(), not just the display: the downloadable run log only keeps printed text.
print(table.round(4).to_string())

## 4. The winner

In [ ]:
WINNER = max(results, key=lambda name: results[name].score)
beats_lr = results[WINNER].score > lr_cv.score
print(
    f"winner: {WINNER} (seed-7 CV {results[WINNER].score:.4f}); "
    f"beats lr ({lr_cv.score:.4f}): {beats_lr}"
)
table.to_csv("/kaggle/working/encoder_select.csv")
print("outputs:", sorted(os.listdir("/kaggle/working")))

if TRACK:
    import wandb

    run = wandb.init(project="name-that-llm", job_type="encoder-select", name="encoder-select")
    run.summary.update({f"cv_{name}": r.score for name, r in results.items()})
    run.summary.update({"cv_lr_seed7": lr_cv.score, "winner": WINNER})
    artifact = wandb.Artifact("encoder-select-oof", type="oof")
    for name in results:
        artifact.add_file(f"/kaggle/working/oof-{name}.npy")
    run.log_artifact(artifact)
    run.finish()

## 5. Findings

From run v1 (2026-10-26, 38 minutes on Kaggle's T4).

**1. ModernBERT wins, and it beats TF-IDF.** On the seed-7 folds:

| model | CV | per fold | minutes |
|---|---|---|---|
| ModernBERT-base (lr 5e-5, batch 16) | **0.344** | 0.326–0.369 | 15 |
| TF-IDF `lr` | 0.330 | | |
| DeBERTa-v3-base (lr 2e-5, batch 8) | 0.206 | 0.161–0.251 | 18 |

P5's criterion, *"an encoder model beats the baseline in CV"*, is met, by a small margin
(+0.014).

**2. DeBERTa underfit, so the comparison isn't settled.** Its training loss barely moved in
3 epochs (about 2.8 down to 2.2; an 18-way guess is 2.89). ModernBERT's fell to 1.15–1.56.
Each model ran at its usual recommended settings, and DeBERTa's (lr 2e-5, batch 8) were too
gentle here. So the tuning notebook first re-runs DeBERTa at ModernBERT's settings, then
tunes whichever of the two scores higher.

**3. Both tokenizers keep the style clues.** DeBERTa's splits "It’s" into `It`, `’`, `s`
and keeps the spaced em dash `▁—` as one token. ModernBERT's byte-level tokenizer keeps
every character too, shown in its raw byte form (`ĠâĢĶ` is a space plus an em dash).

**4. Memory was no problem.** A full batch of the longest passage at 512 tokens peaked at
3.5 GB (DeBERTa, batch 8) and 4.1 GB (ModernBERT, batch 16), out of a T4's 16 GB.

**5. Kaggle gave us two T4 GPUs**, and this notebook used one. Training two folds at once
would halve the time (idea I-019).

**Next:** tune the winner's learning rate, epochs and warmup with Optuna (notebook 08).